# Fast Food Cheat Meal & Workout Burn – Combined Notebook
**Goal:** fulfil the *Summary Feature Selection Matrix* for the two project decisions.

| Feature category | Decision 1 – Exercise plan | Decision 2 – Dietary risk |
|---|---|---|
| Demographics | `fitness_level`, `consumer_age`, `consumer_weight_kg` | not primary |
| Nutritional load | `net_caloric_surplus` | `total_calories`, `daily_calorie_allowance_pct` |
| Macro / micro risks | not primary | `saturated_fat_grams`, `sodium_mg`, `sugar_grams`, `fat_grams`, `carb_grams` |
| Operational targets | `workout_shock_tier`, burn times (running, cycling, walking, hiit) | `meal_category`, `fast_food_chain`, portion reduction flags |

**Sources merged:** `Untitled10(1)` (consumer profile, surplus, meal/chain, reference limits, portion analysis) · `workout_burn_V2` (shock tier, burn times, risk thresholds) · `mm` (personalised recommendation).
Every feature is defined **once** (one load, one `age_group`, one `weight_group`, one set of daily limits).

## 0. Setup, load and shared definitions (run first)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---- ONE load for the whole notebook ----
file_path = "C:/Users/user/Downloads/archive(5)/fast_food_cheat_meal_workout_burn.csv"
df = pd.read_csv(file_path)

# clean column names (mm) and fix the odd first column name 'order_id+A1:Y4' if present (Untitled10)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df.rename(columns={c: "order_id" for c in df.columns if c.startswith("order_id")})
print("Loaded:", df.shape)

# ---- Summary Feature Selection Matrix ----
D1_FEATURES = {
    "Demographics":        ["fitness_level", "consumer_age", "consumer_weight_kg"],
    "Nutritional Load":    ["net_caloric_surplus"],
    "Operational Targets": ["workout_shock_tier", "running_burn_time_min", "cycling_burn_time_min",
                            "walking_burn_time_min", "hiit_burn_time_min"],
}
D2_FEATURES = {
    "Nutritional Load":    ["total_calories", "daily_calorie_allowance_pct"],
    "Macro/Micro Risks":   ["saturated_fat_grams", "sodium_mg", "sugar_grams", "fat_grams", "carb_grams"],
    "Operational Targets": ["meal_category", "fast_food_chain"],   # + portion reduction flags (built in Part C5)
}
needed = [c for d in (D1_FEATURES, D2_FEATURES) for cols in d.values() for c in cols]
missing = [c for c in needed if c not in df.columns]
assert not missing, f"Columns missing from the CSV: {missing}"

# ---- shared constants (defined once) ----
BURN_COLS      = ["walking_burn_time_min", "cycling_burn_time_min", "running_burn_time_min", "hiit_burn_time_min"]
MACRO_COLS     = ["saturated_fat_grams", "sodium_mg", "sugar_grams", "fat_grams", "carb_grams"]
NUTRITION_COLS = ["total_calories", "daily_calorie_allowance_pct"] + MACRO_COLS

AGE_ORDER     = ["18–30", "31–50", "51+"]
WEIGHT_ORDER  = ["<60 kg", "60–80 kg", ">80 kg"]
FITNESS_ORDER = ["Sedentary", "Moderately Active", "Highly Trained"]
TIER_ORDER    = ["Moderate Burn", "High Sweat Penalty", "Severe Cardio Deficit", "Extreme / Half-Marathon Toll"]

# general daily reference values (cite your source in the report: FDA / AHA)
LIMITS = pd.Series({"total_calories": 2000, "fat_grams": 65, "saturated_fat_grams": 20,
                    "carb_grams": 275, "sugar_grams": 50, "sodium_mg": 2300})

# ---- age & weight groups (defined once; same bins as Untitled10) ----
df["age_group"] = pd.cut(df["consumer_age"], bins=[17, 30, 50, np.inf], labels=AGE_ORDER, right=True)
df["weight_group"] = pd.cut(df["consumer_weight_kg"], bins=[-np.inf, 60, 80, np.inf], labels=WEIGHT_ORDER)

### 0.1 Dataset check

In [ ]:
selected = list(dict.fromkeys(needed))
check = pd.DataFrame({"dtype": df[selected].dtypes.astype(str),
                      "missing": df[selected].isna().sum(),
                      "unique": df[selected].nunique()})
display(check)
print("Rows:", df.shape[0], "| Columns:", df.shape[1], "| Duplicate rows:", df.duplicated().sum())
display(df[selected].head())

## Part A. Consumer Profile  (Demographics – Decision 1)
Distribution of `consumer_age` (age group), `consumer_weight_kg` (weight group) and `fitness_level`.

In [ ]:
sns.reset_orig()   # plain matplotlib look for the profile charts

# ---- Age group ----
age_counts = df["age_group"].value_counts().reindex(AGE_ORDER)
plt.figure(figsize=(8, 5))
age_counts.plot(kind="bar")
plt.title("Consumer Distribution by Age Group")
plt.xlabel("Age Group"); plt.ylabel("Number of Consumers"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

# ---- Weight group ----
weight_counts = df["weight_group"].value_counts().reindex(WEIGHT_ORDER)
plt.figure(figsize=(8, 5))
weight_counts.plot(kind="bar")
plt.title("Consumer Distribution by Weight Group")
plt.xlabel("Weight Group"); plt.ylabel("Number of Consumers"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

# ---- Fitness level ----
fitness_counts = df["fitness_level"].value_counts().reindex(FITNESS_ORDER)
plt.figure(figsize=(8, 5))
fitness_counts.plot(kind="bar")
plt.title("Consumer Distribution by Fitness Level")
plt.xlabel("Fitness Level"); plt.ylabel("Number of Consumers"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
profile_summary = pd.concat({
    "Age Group":     pd.DataFrame({"Consumers": age_counts,     "Percentage": age_counts / len(df) * 100}),
    "Weight Group":  pd.DataFrame({"Consumers": weight_counts,  "Percentage": weight_counts / len(df) * 100}),
    "Fitness Level": pd.DataFrame({"Consumers": fitness_counts, "Percentage": fitness_counts / len(df) * 100}),
})
display(profile_summary.round(2))

## Part B. Decision 1 – Exercise Plan
Features: `fitness_level`, `consumer_age`, `consumer_weight_kg`, `net_caloric_surplus`, `workout_shock_tier`, burn times.

### B1. Nutritional load: `net_caloric_surplus`

In [ ]:
surplus = df["net_caloric_surplus"]
print(f"Mean: {surplus.mean():,.1f} | Median: {surplus.median():,.1f} | Min: {surplus.min():,.1f} | Max: {surplus.max():,.1f}")

# surplus by each demographic variable
for group_col in ["age_group", "weight_group", "fitness_level"]:
    print(f"\n--- Net caloric surplus by {group_col} ---")
    display(df.groupby(group_col, observed=True).agg(
        Consumers=("net_caloric_surplus", "count"),
        Mean_Surplus=("net_caloric_surplus", "mean"),
        Median_Surplus=("net_caloric_surplus", "median"),
        Mean_Calories=("total_calories", "mean")).round(2))

# age x weight matrix for each fitness level
for fitness in FITNESS_ORDER:
    print(f"\n--- Mean net caloric surplus: {fitness} ---")
    display(df[df["fitness_level"] == fitness]
            .pivot_table(values="net_caloric_surplus", index="age_group", columns="weight_group",
                         aggfunc="mean", observed=False)
            .reindex(index=AGE_ORDER, columns=WEIGHT_ORDER).round(2))

### B2. Operational target: `workout_shock_tier`

In [ ]:
# counts and % of each tier within each fitness level
tier_counts = pd.crosstab(df["fitness_level"], df["workout_shock_tier"]).reindex(index=FITNESS_ORDER, columns=TIER_ORDER)
print("--- Tier counts by fitness level ---"); display(tier_counts)
print("--- Tier distribution by fitness level (%) ---"); display((tier_counts.div(tier_counts.sum(axis=1), axis=0) * 100).round(2))
print("--- Most frequent tier per fitness level ---"); display(tier_counts.idxmax(axis=1).to_frame("most_frequent_tier"))

# fitness x age group
idx = pd.MultiIndex.from_product([FITNESS_ORDER, AGE_ORDER], names=["fitness_level", "age_group"])
ct = (pd.crosstab([df["fitness_level"], df["age_group"].astype(str)], df["workout_shock_tier"])
        .reindex(index=idx, columns=TIER_ORDER, fill_value=0))
print("--- Most frequent tier by fitness level and age group ---")
display(ct.idxmax(axis=1).unstack().reindex(columns=AGE_ORDER))

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", cbar=True)
plt.title("Bivariate Frequency: Workout Shock Tier by Fitness Level & Age Group", fontweight="bold")
plt.xlabel("Workout Shock Tier"); plt.ylabel("Fitness Level & Age Group")
plt.tight_layout(); plt.show()

### B3. Operational targets: burn times (walking, cycling, running, HIIT)

In [ ]:
d1_num_cols = ["consumer_age", "consumer_weight_kg", "net_caloric_surplus"] + BURN_COLS
d1_summary = df[d1_num_cols].describe().T[["count", "mean", "std", "min", "50%", "max"]]
d1_summary.columns = ["Count", "Mean", "Std Dev", "Min", "Median (50%)", "Max"]
print("--- Decision 1 numerical attributes ---"); display(d1_summary.round(2))
print("--- Average burn time (min) by fitness level ---"); display(df.groupby("fitness_level")[BURN_COLS].mean().reindex(FITNESS_ORDER).round(2))
print("--- Average burn time (min) by workout shock tier ---"); display(df.groupby("workout_shock_tier")[BURN_COLS].mean().reindex(TIER_ORDER).round(2))
print("--- Average burn time (min) by age group ---"); display(df.groupby("age_group", observed=True)[BURN_COLS].mean().round(2))
print("--- Average burn time (min) by weight group ---"); display(df.groupby("weight_group", observed=True)[BURN_COLS].mean().round(2))

In [ ]:
sns.set_theme(style="whitegrid")
fig1, axes1 = plt.subplots(1, 2, figsize=(14, 5))

# 1A. burn duration by modality and shock tier
burn_means = df.groupby("workout_shock_tier")[BURN_COLS].mean().reindex(TIER_ORDER)
burn_means.plot(kind="bar", ax=axes1[0], colormap="viridis", width=0.8)
axes1[0].set_title("Decision 1: Average Burn Duration by Workout Shock Tier", fontweight="bold", fontsize=11)
axes1[0].set_xlabel("Workout Shock Tier", fontweight="bold"); axes1[0].set_ylabel("Duration (Minutes)", fontweight="bold")
axes1[0].legend(["Walking", "Cycling", "Running", "HIIT"], title="Exercise Modality")
axes1[0].tick_params(axis="x", rotation=15)

# 1B. running time by fitness level and weight group (same weight_group as everywhere else)
sns.barplot(data=df, x="fitness_level", y="running_burn_time_min", hue="weight_group",
            order=FITNESS_ORDER, hue_order=WEIGHT_ORDER, palette="magma", ax=axes1[1])
axes1[1].set_title("Decision 1: Running Burn Time by Fitness Level & Weight Group", fontweight="bold", fontsize=11)
axes1[1].set_xlabel("Fitness Level", fontweight="bold"); axes1[1].set_ylabel("Running Time Required (Mins)", fontweight="bold")
axes1[1].set_ylim(0, 215)   # head-room so the legend does not cover the bars
axes1[1].legend(title="Weight Group", ncol=3, loc="upper center", fontsize=9)

plt.tight_layout()
plt.savefig("decision1_exercise_plan.png", dpi=300)
plt.show()

#### B3.1 Numbers behind chart 1B
Mean running time and number of orders for every fitness level × weight group (same `weight_group` as the profile charts: <60 kg, 60–80 kg, >80 kg). Cite these exact values in the report.

In [ ]:
running_tbl = (df.pivot_table(values="running_burn_time_min", index="fitness_level", columns="weight_group",
                              aggfunc="mean", observed=False)
                 .reindex(index=FITNESS_ORDER, columns=WEIGHT_ORDER))
orders_tbl = (pd.crosstab(df["fitness_level"], df["weight_group"])
                .reindex(index=FITNESS_ORDER, columns=WEIGHT_ORDER))

print("--- Mean running burn time (min) by fitness level and weight group ---")
display(running_tbl.round(1))
print("--- Number of orders in each cell ---")
display(orders_tbl)

### B4. Personalised exercise recommendation (from `mm`)
Uses the dataset's own `workout_shock_tier` and burn-time columns, so no tier or burn-time feature is calculated twice.

In [ ]:
dataset_avg_surplus = df["net_caloric_surplus"].mean()
df["exceeds_avg_surplus"] = df["net_caloric_surplus"] > dataset_avg_surplus
print(f"Orders above the average surplus ({dataset_avg_surplus:,.1f} kcal): {df['exceeds_avg_surplus'].mean() * 100:.1f}%")

def get_workout_recommendation(row, dataset_avg=dataset_avg_surplus):
    surplus = row["net_caloric_surplus"]
    run, cycle, walk, hiit = (row["running_burn_time_min"], row["cycling_burn_time_min"],
                              row["walking_burn_time_min"], row["hiit_burn_time_min"])
    print("=" * 60)
    print(f"USER PROFILE: Age: {row['consumer_age']} | Weight: {row['consumer_weight_kg']}kg | Fitness Level: {row['fitness_level']}")
    print(f"Meal Calorie Surplus: {surplus:,.1f} kcal")
    print(f"Dataset Average Surplus: {dataset_avg:,.1f} kcal")
    print(f"Workout Shock Tier: {row['workout_shock_tier']}")
    print("-" * 60)
    if surplus > dataset_avg:
        print(f"⚠️ သင့်ရဲ့ Net Calorie Surplus ({surplus:,.1f} kcal) ဟာ ပျမ်းမျှ ({dataset_avg:,.1f} kcal) ထက် ပိုများနေပါတယ်။")
        print("သင့်အတွက် အကြံပြုထားသော လေ့ကျင့်ခန်း ကြာချိန်များ -")
        print(f" ⚡ HIIT    (Very High Intensity): {hiit:.1f} မိနစ် (အချိန် ကုန်သက်သာဆုံး)")
        print(f" 🏃 Running (High Intensity): {run:.1f} မိနစ်")
        print(f" 🚴 Cycling (Moderate Intensity): {cycle:.1f} မိနစ်")
        print(f" 🚶 Walking (Low Intensity): {walk:.1f} မိနစ်")
    else:
        print(f"✅ သင့်ရဲ့ Net Calorie Surplus ({surplus:,.1f} kcal) ဟာ ပျမ်းမျှထက် နည်းပါသေးတယ်။")
        print(f"ပုံမှန် လေ့ကျင့်ခန်းများ (Running: {run:.1f} / Cycling: {cycle:.1f} / Walking: {walk:.1f} / HIIT: {hiit:.1f} မိနစ်) ပြုလုပ်ပေးရုံဖြင့် လုံလောက်ပါတယ်။")
    print("=" * 60)

get_workout_recommendation(df.iloc[0])

In [ ]:
# Sanity check (nothing is stored in df): does the MET formula used in mm reproduce the dataset's burn times?
for name, met in {"walking": 3.8, "cycling": 7.5, "running": 9.8}.items():
    est = df["net_caloric_surplus"] / ((met * 3.5 * df["consumer_weight_kg"]) / 200)
    actual = df[f"{name}_burn_time_min"]
    print(f"{name:8s} MET={met}: correlation = {est.corr(actual):.3f} | mean difference = {(actual - est).mean():+.1f} min | ratio dataset/MET = {(actual / est).mean():.2f}")

## Part C. Decision 2 – Dietary Risk & Portion Control
Features: `total_calories`, `daily_calorie_allowance_pct`, `saturated_fat_grams`, `sodium_mg`, `sugar_grams`, `fat_grams`, `carb_grams`, `meal_category`, `fast_food_chain`, portion reduction flags.

### C1. Nutritional load and macro/micro summary

In [ ]:
d2_summary = df[NUTRITION_COLS].describe().T[["count", "mean", "std", "min", "50%", "max"]]
d2_summary.columns = ["Count", "Mean", "Std Dev", "Min", "Median (50%)", "Max"]
display(d2_summary.round(2))

### C2. Observed averages vs daily reference values

In [ ]:
ref_cols = list(LIMITS.index)
comparison = pd.DataFrame({"Observed Mean (1 meal)": df[ref_cols].mean(), "Daily Reference": LIMITS})
comparison["% of Daily Reference"] = comparison["Observed Mean (1 meal)"] / comparison["Daily Reference"] * 100
comparison["Exceeds Reference"] = comparison["Observed Mean (1 meal)"] > comparison["Daily Reference"]
display(comparison.sort_values("% of Daily Reference", ascending=False).round(2))

### C3. Risk threshold exceedance (per order)

In [ ]:
# one flag per feature in the matrix: does ONE meal exceed the whole day's reference value?
for col in MACRO_COLS:
    df[f"exceed_{col}"] = df[col] > LIMITS[col]
df["exceed_100pct_cal"] = df["daily_calorie_allowance_pct"] >= 100
df["compound_high_risk"] = df["exceed_saturated_fat_grams"] & df["exceed_sodium_mg"]

flag_cols = [f"exceed_{c}" for c in MACRO_COLS] + ["exceed_100pct_cal", "compound_high_risk"]
risk_summary = pd.DataFrame({"Affected Orders": df[flag_cols].sum(), "Prevalence (%)": df[flag_cols].mean() * 100})
display(risk_summary.round(2))

print("--- Exceedance rate (%) by meal_category ---")
display((df.groupby("meal_category")[flag_cols].mean() * 100).round(1))
print("--- Exceedance rate (%) by fast_food_chain ---")
display((df.groupby("fast_food_chain")[flag_cols].mean() * 100).round(1))

### C3.1 Micro/Macro health risk matrix (scatter plot)
Saturated fat (x) against sodium (y) for every order, coloured by `daily_calorie_allowance_pct`.
The two reference lines are the daily limits (20 g saturated fat, 2,300 mg sodium), so the chart is split into four quadrants:
lower-left = within both limits, upper-right = over both limits.

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# Scatter plot: Saturated Fat vs Sodium, coloured by Calorie Allowance %
scatter = plt.scatter(
    df["saturated_fat_grams"],
    df["sodium_mg"],
    c=df["daily_calorie_allowance_pct"],
    cmap="YlOrRd",
    alpha=0.6
)

# Colour bar for the calorie allowance scale
cbar = plt.colorbar(scatter)
cbar.set_label("Daily Calorie Allowance %", fontweight="bold")

# Reference lines = maximum daily health limits (taken from LIMITS, defined once in the setup cell)
plt.axhline(LIMITS["sodium_mg"], color="darkred", linestyle=":",
            label=f"Max Daily Sodium Limit ({LIMITS['sodium_mg']:,.0f} mg)")
plt.axvline(LIMITS["saturated_fat_grams"], color="darkred", linestyle="--",
            label=f"Max Daily Sat. Fat Limit ({LIMITS['saturated_fat_grams']:.0f} g)")

# Labels, title and legend
plt.title("Decision 2: Micro/Macro Health Risk Matrix", fontweight="bold", fontsize=12)
plt.xlabel("Saturated Fat (Grams)", fontweight="bold")
plt.ylabel("Sodium (mg)", fontweight="bold")
plt.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.savefig("decision2_health_risk_matrix.png", dpi=300)
plt.show()

In [ ]:
# How many orders fall in each quadrant of the scatter plot?
over_sat = df["saturated_fat_grams"] > LIMITS["saturated_fat_grams"]
over_na  = df["sodium_mg"] > LIMITS["sodium_mg"]

quadrants = pd.DataFrame({
    "Orders": [(over_sat & over_na).sum(), (over_sat & ~over_na).sum(),
               (~over_sat & over_na).sum(), (~over_sat & ~over_na).sum()]},
    index=["Upper-right: over BOTH limits", "Lower-right: over saturated fat only",
           "Upper-left: over sodium only", "Lower-left: within BOTH limits"])
quadrants["Percentage"] = quadrants["Orders"] / len(df) * 100
display(quadrants.round(2))

print("Max saturated fat:", df["saturated_fat_grams"].max(), "g | orders at that value:", (df["saturated_fat_grams"] == df["saturated_fat_grams"].max()).sum())
print("Max sodium:", df["sodium_mg"].max(), "mg | orders at that value:", (df["sodium_mg"] == df["sodium_mg"].max()).sum())

### C4. Operational targets: `meal_category` and `fast_food_chain`

In [ ]:
def category_profile(col):
    counts = df[col].value_counts()
    summary = pd.DataFrame({"Orders": counts, "Percentage": counts / len(df) * 100})
    means = df.groupby(col)[NUTRITION_COLS].mean()
    spread = (means.max() - means.min()).to_frame("Highest - Lowest Difference").sort_values("Highest - Lowest Difference", ascending=False)
    print(f"\n======== {col} ========")
    print("Orders:");                              display(summary.round(2))
    print("Mean nutrition per order:");            display(means.round(2))
    print("How much do categories differ?");       display(spread.round(2))

category_profile("meal_category")
category_profile("fast_food_chain")

### C5. Portion reduction flags
For each order we test the 100 %, 75 % and 50 % portion against the daily limits for saturated fat and sodium and against 100 % of the daily calorie allowance. This is a strict test (one meal compared with a whole day's limit).

In [ ]:
portion_levels = {"100%": 1.00, "75%": 0.75, "50%": 0.50}

# average nutrition if the portion is reduced
portion_analysis = pd.DataFrame({name: df[ref_cols].mean() * p for name, p in portion_levels.items()})
print("--- Average nutrition by portion size ---"); display(portion_analysis.round(2))
print("--- % of daily reference by portion size ---"); display((portion_analysis.div(LIMITS, axis=0) * 100).round(1))

# per-order flags
def within_limits(p):
    return ((df["saturated_fat_grams"] * p <= LIMITS["saturated_fat_grams"]) &
            (df["sodium_mg"] * p <= LIMITS["sodium_mg"]) &
            (df["daily_calorie_allowance_pct"] * p < 100))

for name, p in portion_levels.items():
    df[f"safe_at_{name.strip('%')}pct"] = within_limits(p)

df["portion_reduction_flag"] = np.select(
    [df["safe_at_100pct"], df["safe_at_75pct"], df["safe_at_50pct"]],
    ["No reduction needed", "Reduce to 75%", "Reduce to 50%"],
    default="Still exceeds at 50%")

FLAG_ORDER = ["No reduction needed", "Reduce to 75%", "Reduce to 50%", "Still exceeds at 50%"]
flag_counts = df["portion_reduction_flag"].value_counts().reindex(FLAG_ORDER).fillna(0)
display(pd.DataFrame({"Orders": flag_counts, "Percentage": flag_counts / len(df) * 100}).round(2))

print("--- Portion reduction flag by meal_category (% of orders) ---")
display((pd.crosstab(df["meal_category"], df["portion_reduction_flag"], normalize="index") * 100)
        .reindex(columns=FLAG_ORDER, fill_value=0).round(1))

#### C5.1 Chart: portion reduction flag by meal category and fast-food chain
Each bar adds up to 100 % of that category's orders. The red label is the share that still breaks a daily limit even after halving the portion.

In [ ]:
sns.set_theme(style="whitegrid")
FLAG_COLORS = {"No reduction needed": "#59A14F", "Reduce to 75%": "#F2CF5B",
               "Reduce to 50%": "#F28E2B", "Still exceeds at 50%": "#D62728"}

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharex=True)
for ax, col, title in zip(axes, ["meal_category", "fast_food_chain"], ["Meal Category", "Fast-Food Chain"]):
    pct = (pd.crosstab(df[col], df["portion_reduction_flag"], normalize="index") * 100
           ).reindex(columns=FLAG_ORDER, fill_value=0).sort_values("Still exceeds at 50%")
    pct.plot(kind="barh", stacked=True, ax=ax, width=0.8, edgecolor="white",
             color=[FLAG_COLORS[f] for f in FLAG_ORDER], legend=False)
    # label the share of orders that still exceed limits at a 50% portion
    left = pct[FLAG_ORDER[:-1]].sum(axis=1)
    for y, (l, v) in enumerate(zip(left, pct["Still exceeds at 50%"])):
        if v >= 3:
            ax.text(l + v / 2, y, f"{v:.0f}%", ha="center", va="center", color="white", fontweight="bold", fontsize=9)
    ax.set_title(f"Decision 2: Portion Reduction Flag by {title}", fontweight="bold", fontsize=11)
    ax.set_xlabel("Share of orders (%)", fontweight="bold"); ax.set_ylabel(title, fontweight="bold")
    ax.set_xlim(0, 100)

handles = [plt.Rectangle((0, 0), 1, 1, color=FLAG_COLORS[f]) for f in FLAG_ORDER]
fig.legend(handles, FLAG_ORDER, title="Portion reduction flag", loc="lower center", ncol=4, frameon=False)
plt.tight_layout(rect=[0, 0.1, 1, 1])
plt.savefig("decision2_portion_reduction.png", dpi=300)
plt.show()

### C6. Personalised dietary recommendation

In [ ]:
def get_dietary_recommendation(row):
    print("=" * 60)
    print(f"MEAL: {row['meal_category']} | {row['fast_food_chain']}")
    print(f"Calories: {row['total_calories']:,.0f} kcal ({row['daily_calorie_allowance_pct']:.0f}% of daily allowance)")
    for col, label, unit in [("saturated_fat_grams", "Saturated fat", "g"), ("sodium_mg", "Sodium", "mg"),
                             ("sugar_grams", "Sugar", "g"), ("fat_grams", "Fat", "g"), ("carb_grams", "Carbs", "g")]:
        pct = row[col] / LIMITS[col] * 100
        print(f"  {label:14s}: {row[col]:,.0f} {unit}  ({pct:.0f}% of daily limit)")
    print("-" * 60)
    flag = row["portion_reduction_flag"]
    if flag == "No reduction needed":
        print("✅ ဤအစားအစာသည် နေ့စဉ်ကန့်သတ်ချက်အတွင်း ရှိပါသည်။")
    elif flag == "Still exceeds at 50%":
        print("⚠️ ပမာဏတဝက်လျှော့ပေမယ့် ကန့်သတ်ချက်ကို ကျော်နေဆဲပါ။ ပိုပေါ့တဲ့ menu သို့ ပြောင်းရန် အကြံပြုပါသည်။")
    else:
        print(f"⚠️ အကြံပြုချက်: {flag}")
    print("=" * 60)

get_dietary_recommendation(df.iloc[0])

## Part D. Feature selection matrix – coverage check

In [ ]:
coverage = pd.DataFrame([
    ("Demographics",        "fitness_level, consumer_age, consumer_weight_kg", "Decision 1", "Part A, B1, B3, B4"),
    ("Nutritional Load",    "net_caloric_surplus",                             "Decision 1", "B1, B4"),
    ("Operational Targets", "workout_shock_tier",                              "Decision 1", "B2, B3, B4"),
    ("Operational Targets", "running / cycling / walking / hiit burn times",   "Decision 1", "B3, B4"),
    ("Nutritional Load",    "total_calories, daily_calorie_allowance_pct",     "Decision 2", "C1, C2, C3, C3.1"),
    ("Macro/Micro Risks",   "saturated_fat, sodium, sugar, fat, carb",         "Decision 2", "C1, C2, C3, C3.1, C5"),
    ("Operational Targets", "meal_category, fast_food_chain",                  "Decision 2", "C3, C4, C5"),
    ("Operational Targets", "portion reduction flags",                         "Decision 2", "C5, C5.1, C6"),
], columns=["Feature category", "Features", "Decision", "Where in this notebook"])
display(coverage)

built = ["age_group", "weight_group", "exceeds_avg_surplus", "portion_reduction_flag", "compound_high_risk"]
print("All matrix columns present:", all(c in df.columns for c in needed))
print("Derived features present :", all(c in df.columns for c in built))